# Does rare-word ratio improve P2?

Both models predict whether randomized side A wins a published pair using signed A-minus-B feature differences.

| Model | Opening features |
| --- | --- |
| P2 | Word count and mean content-word Zipf |
| P2 + rare ratio | P2 features plus the share of all listed word occurrences with Zipf < 3 |

The added feature is the [primary rare-word definition](../lexical-complexity/rare_word_ratio.ipynb). Each model uses L2 logistic regression with `C=1`, standardization fitted inside each training fold, and the same three folds grouped by OP discussion. This comparison uses out-of-fold predictions on published training pairs. Held-out generalization remains a separate evaluation.

[Original P2 script](train_p2_word_count_and_content_word_zipf_differences.py) · [P2 plus rare-word ratio script](train_p2_plus_all_word_rare_ratio_zipf_below_3.py)

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

REPO = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "team-setup").is_dir() and (p / "pyproject.toml").is_file()
)
ROOT = REPO / "project/data/processed/linguistic-features/paired-models"
MODELS = {
    "P2": ("p2_word_count_and_content_word_zipf_differences", "20261120T201405034740Z"),
    "P2 + rare ratio": ("p2_plus_all_word_rare_ratio_zipf_below_3", "20261120T204255709584Z"),
}
KEYS = ["pair_id", "post_id", "a_comment_id", "b_comment_id", "a_wins", "fold",
        "a_minus_b_word_count", "a_minus_b_mean_content_word_zipf"]

reports = {}
frames = {}
for label, (folder, run_id) in MODELS.items():
    run = ROOT / folder / run_id
    reports[label] = json.loads((run / "results.json").read_text())
    frames[label] = pd.read_parquet(run / "oof_predictions.parquet")
baseline, expanded = frames.values()
pd.testing.assert_frame_equal(baseline[KEYS], expanded[KEYS])
assert reports["P2"]["input_sha256"] == reports["P2 + rare ratio"]["input_sha256"]
assert expanded["pair_id"].is_unique
assert expanded.groupby("post_id")["fold"].nunique().eq(1).all()

data = expanded.drop(columns="oof_probability_a_wins").copy()
for label, frame in frames.items():
    data[label] = frame["oof_probability_a_wins"]
print(f"{len(data):,} pairs across {data['post_id'].nunique():,} discussions")

3,451 pairs across 2,507 discussions


## Out-of-fold performance and paired uncertainty

ROC-AUC and average precision evaluate ranking. Pairwise accuracy chooses A at probability 0.5 or higher. Lower log loss indicates better probability estimates.

For each of 1,000 bootstrap draws, resample whole OP discussions and apply the same weights to both models' fixed out-of-fold predictions. The 95% intervals describe variation across discussions in this comparison.

In [2]:
from sklearn.metrics import accuracy_score, average_precision_score, log_loss, roc_auc_score


def pairwise_accuracy(y, probability, sample_weight=None):
    return accuracy_score(y, probability >= 0.5, sample_weight=sample_weight)


metric_functions = {
    "ROC-AUC": roc_auc_score,
    "Average precision": average_precision_score,
    "Pairwise accuracy": pairwise_accuracy,
    "Log loss": log_loss,
}
y = data["a_wins"].to_numpy()
p2, expanded_probability = data[list(MODELS)].to_numpy().T
post_codes, posts = pd.factorize(data["post_id"])
rng = np.random.default_rng(42)
differences = {name: [] for name in metric_functions}
for _ in range(1000):
    post_weights = rng.multinomial(len(posts), np.full(len(posts), 1 / len(posts)))
    weights = post_weights[post_codes]
    for name, metric in metric_functions.items():
        differences[name].append(
            metric(y, expanded_probability, sample_weight=weights)
            - metric(y, p2, sample_weight=weights)
        )

rows = []
for name, metric in metric_functions.items():
    base, extra = metric(y, p2), metric(y, expanded_probability)
    low, high = np.quantile(differences[name], [0.025, 0.975])
    rows.append({
        "Metric": name, "P2": base, "P2 + rare ratio": extra, "Change": extra - base,
        "95% CI lower": low, "95% CI upper": high,
    })
performance = pd.DataFrame(rows).set_index("Metric")
display(performance.round(5))

,P2,P2 + rare ratio,Change,95% CI lower,95% CI upper
Metric,,,,,
ROC-AUC,0.70324,0.70292,-0.00032,-0.00317,0.00262
Average precision,0.69426,0.69362,-0.00064,-0.00326,0.00232
Pairwise accuracy,0.64677,0.64474,-0.00203,-0.00864,0.00488
Log loss,0.62701,0.62731,0.00030,-0.00132,0.00185


## Pearson and Spearman correlations

Pearson measures linear association across signed A-minus-B differences. Spearman measures monotonic association using ranks and is less sensitive to extreme feature values.

In [3]:
feature_labels = {
    "a_minus_b_word_count": "Word count",
    "a_minus_b_mean_content_word_zipf": "Mean content-word Zipf",
    "a_minus_b_rare_word_ratio": "All-word rare ratio (Zipf < 3)",
}
features = data[list(feature_labels)].rename(columns=feature_labels)

pearson = features.corr(method="pearson")
spearman = features.corr(method="spearman")

print("Pearson correlation")
display(pearson.round(4))
print("Spearman correlation")
display(spearman.round(4))

Pearson correlation


,Word count,Mean content-word Zipf,All-word rare ratio (Zipf < 3)
Word count,1.0000,-0.0534,0.0002
Mean content-word Zipf,-0.0534,1.0000,-0.5204
All-word rare ratio (Zipf < 3),0.0002,-0.5204,1.0000


Spearman correlation


,Word count,Mean content-word Zipf,All-word rare ratio (Zipf < 3)
Word count,1.0000,-0.0571,0.0361
Mean content-word Zipf,-0.0571,1.0000,-0.4915
All-word rare ratio (Zipf < 3),0.0361,-0.4915,1.0000


## Method verification

Reproduce the fold predictions and seeded A/B assignments. Check source fingerprints, training scope, and the rarity cutoff.

In [4]:
import hashlib
import sys

sys.path.insert(0, str(REPO / "project/notebooks/linguistic-features"))
from paired_evaluation import SEED, make_paired_classifier

# Refit the exact shared pipeline and check each scaler against its fitting rows.
scalers_checked = 0
for label, report in reports.items():
    features = report["features"]
    for fold in sorted(data["fold"].unique()):
        validation = data["fold"].eq(fold)
        fitting = ~validation
        assert set(data.loc[fitting, "post_id"]).isdisjoint(data.loc[validation, "post_id"])
        train_features = data.loc[fitting, features]
        pipeline = make_paired_classifier().fit(train_features, data.loc[fitting, "a_wins"])
        scaler = pipeline.named_steps["standardscaler"]
        assert scaler.n_samples_seen_ == len(train_features)
        np.testing.assert_allclose(scaler.mean_, train_features.mean().to_numpy())
        np.testing.assert_allclose(scaler.var_, train_features.var(ddof=0).to_numpy())
        np.testing.assert_allclose(
            pipeline.predict_proba(data.loc[validation, features])[:, 1], data.loc[validation, label],
            rtol=1e-10, atol=1e-12,
        )
        scalers_checked += 1

# Reproduce the seeded assignments from the full pinned training-pair table.
assignments = pd.read_parquet(ROOT.parent / "training_pair_assignments.parquet").sort_values("pair_id")
expected_sides = np.random.default_rng(SEED).integers(0, 2, len(assignments))
np.testing.assert_array_equal(assignments["a_wins"].to_numpy(), expected_sides)
saved = assignments.set_index("pair_id").loc[data["pair_id"], ["post_id", "a_wins", "fold"]]
pd.testing.assert_frame_equal(data.set_index("pair_id")[saved.columns], saved)
a_win_count = int(data["a_wins"].sum())

# Each bootstrap draw assigns one multiplicity per discussion, shared by both models.
assert all(len(draws) == 1000 for draws in differences.values())
assert post_weights.sum() == len(posts)
assert pd.Series(weights).groupby(data["post_id"].to_numpy()).nunique().eq(1).all()

# Verify both feature sources and check the rare-word difference against the paired source.
def verify_training_input(folder, run_id, fingerprint):
    source_dir = ROOT.parent / folder / run_id
    source_report = json.loads((source_dir / "results.json").read_text())
    assert source_report["split"] == "train"
    path = source_dir / "paired_scores.parquet"
    with path.open("rb") as source:
        assert hashlib.file_digest(source, "sha256").hexdigest() == fingerprint
    return source_report, path


for report in reports.values():
    assert report["split"] == "train"
    verify_training_input("word-frequency-sensitivity", report["input_run"], report["input_sha256"])
extra = reports["P2 + rare ratio"]
rare_report, rare_path = verify_training_input(
    "rare-word-ratio", extra["rare_word_input_run"], extra["rare_word_input_sha256"]
)
assert rare_report["rare_zipf_cutoff_exclusive"] == 3
rare = pd.read_parquet(rare_path).set_index("pair_id").loc[data["pair_id"]]
np.testing.assert_array_equal(
    (rare["a_rare_word_ratio"] - rare["b_rare_word_ratio"]).to_numpy(),
    data["a_minus_b_rare_word_ratio"].to_numpy(),
)
print(f"Reproduced {scalers_checked} fold-specific scalers and their saved predictions.")
print(f"Verified discussion-grouped folds, seed {SEED} A/B assignments ({a_win_count:,} A wins),")
print("1,000 paired discussion-bootstrap draws, and training-only feature inputs.")

Reproduced 6 fold-specific scalers and their saved predictions.
Verified discussion-grouped folds, seed 42 A/B assignments (1,714 A wins),
1,000 paired discussion-bootstrap draws, and training-only feature inputs.
